### Compile to Hailo HEF
Run this when ready to compile for a Hailo device; training and validation are not required. Ultralytics uses the selected dataset YAML as INT8 calibration data. To run, you'll first need to choose the accelerator that will run the HEF. The accelerator you choose *must* be the one that runs the HEF; any other device will reject it. Additionally, ensure that your calibration dataset is representative of your deployment environment, as the HEF will calibrate against its images during compilation.

The Hailo Dataflow Compiler (DFC) matching the selected accelerator must be installed and licensed in the export environment. Ultralytics Hailo compilation requires Linux x86_64; it cannot compile on macOS, Windows, or ARM. For more info on model compilation, see the [Ultralytics Hailo export guide](https://docs.ultralytics.com/integrations/hailo/).

The export cell can be run on its own: enter the path to trained model weights and a calibration `data.yaml`. If you have already trained a model or downloaded the dataset in this notebook, their paths are filled in automatically when available.

In [ ]:
from pathlib import Path

import ipywidgets as w
from IPython.display import display
from ultralytics import YOLO

trainer = getattr(globals().get("yolo"), "trainer", None)
trained_model_path = (
    Path(trainer.save_dir) / "weights" / "best.pt"
    if trainer is not None
    else ""
)
current_dataset = globals().get("dataset")
calibration_data_path = (
    Path(current_dataset.location) / "data.yaml"
    if current_dataset is not None and getattr(current_dataset, "location", None)
    else ""
)
current_imgsz = globals().get("imgsz_w")
default_imgsz = getattr(current_imgsz, "value", 640)

hailo_target = w.Dropdown(
    description="Target accelerator:",
    options=["hailo8", "hailo8l", "hailo10h", "hailo15h", "hailo15l"],
    value="hailo10h",
    style={"description_width": "160px"},
)
trained_model = w.Text(
    description="Model weights:",
    placeholder="Path to trained .pt weights",
    value=str(trained_model_path),
    style={"description_width": "160px"},
    layout=w.Layout(width="800px"),
)
calibration_data = w.Text(
    description="Calibration YAML:",
    placeholder="Path to dataset data.yaml",
    value=str(calibration_data_path),
    style={"description_width": "160px"},
    layout=w.Layout(width="800px"),
)
imgsz_input = w.BoundedIntText(
    description="Image size:",
    value=default_imgsz,
    min=32,
    max=2048,
    step=32,
    style={"description_width": "160px"},
)
export_button = w.Button(description="Compile HEF", button_style="success")
export_output = w.Output()

def export_hailo_hef(_):
    with export_output:
        export_output.clear_output()
        best_weights = Path(trained_model.value).expanduser()
        if not trained_model.value.strip():
            raise ValueError("Enter the path to trained model weights.")
        if not calibration_data.value.strip():
            raise ValueError("Enter the path to the calibration dataset data.yaml.")

        calibration_yaml = Path(calibration_data.value).expanduser()
        if not calibration_yaml.is_file():
            raise FileNotFoundError(f"Calibration dataset YAML not found: {calibration_yaml}")
        if not best_weights.is_file():
            raise FileNotFoundError(f"Trained model weights not found: {best_weights}")

        best_model = YOLO(str(best_weights))
        hef_path = best_model.export(
            format="hailo",
            name=hailo_target.value,
            data=str(calibration_yaml),
            imgsz=imgsz_input.value,
        )
        print(f"Hailo export output: {hef_path}")

export_button.on_click(export_hailo_hef)
display(w.VBox([trained_model, calibration_data, imgsz_input, hailo_target, export_button, export_output]))